# **Processamento Inicial de Dados da API do The Cancer Genome Atlas (TCGA)**
Desenvolvida pela Genomic Data Commons (GDC)

# Importação de Bibliotecas

In [1]:
import json
import pandas as pd
import requests

# Constantes

In [2]:
# URL HTML base da API TCGA / GDC
TCGA_API_ENDPOINT = 'https://api.gdc.cancer.gov'

# Endpoint de casos relacionados aos arquivos
CASES_ENDPOINT = f'{TCGA_API_ENDPOINT}/cases'

# Endpoint usado para download dos arquivos
DATA_ENDPOINT = f'{TCGA_API_ENDPOINT}/data'

# Endpoint de arquivos relacionados aos casos
FILES_ENDPOINT = f'{TCGA_API_ENDPOINT}/files'

# Endpoint de projetos relacionados aos programas
PROJECTS_ENDPOINT = f'{TCGA_API_ENDPOINT}/projects'

# Endpoint para slicing dos arquivos BAM
SLICING_ENDPOINT = f'{TCGA_API_ENDPOINT}/slicing'

# Endpoint de status e versão da API
STATUS_ENDPOINT = f'{TCGA_API_ENDPOINT}/status'

# Status da API

In [3]:
# Verifica o status e a versão da API
response = requests.get(STATUS_ENDPOINT)

# Printa o conteúdo da resposta
print(response.content.decode('utf-8'))

{"commit":"3ed4235efe2f2d9c9969e1ce790e6b34a5c60c9d","data_release":"Data Release 41.0 - August 28, 2024","status":"OK","tag":"7.4.0","version":1}



# Processamento

## Total de Projetos

In [4]:
# Mapeia todas as informações contidas no endpoint
response = requests.get(f'{PROJECTS_ENDPOINT}/_mapping')

# Converte o conteúdo da resposta para JSON
json_response = json.loads(response.content.decode('utf-8'))

# Armazena os campos contidos no endpoint
fields_projects = json.dumps(json_response['fields'])[1:-2]
fields_projects = fields_projects.replace('"', '')
fields_projects = fields_projects.replace(' ', '')

In [5]:
# Busca por todos os objetos do endpoint
response = requests.post(
    url=PROJECTS_ENDPOINT,
    headers={'Content-Type': 'application/json'},
    json={'fields': 'name'}
)

# Converte o conteúdo da resposta para JSON
json_response = json.loads(response.content.decode('utf-8'))

# Armazena total de projetos contidos no TCGA
total_projects = json.dumps(json_response['data']['pagination']['total'])
print(f'Total de projetos no TCGA: {total_projects}')

Total de projetos no TCGA: 86


In [6]:
# Parâmetros para a requisição ao endpoint
params = {
    'fields': fields_projects,
    'size': total_projects
}

# Busca por todos os objetos do endpoint
response = requests.post(
    url=PROJECTS_ENDPOINT,
    headers={'Content-Type': 'application/json'},
    json=params
)

# Converte o conteúdo da resposta para JSON
json_response = json.loads(response.content.decode('utf-8'))['data']['hits']

# Transforma os valores booleanos do JSON em strings
for index in range(len(json_response)):
    obj = json_response[index]
    for key in obj:
        if isinstance(obj[key], bool): 
            obj[key] = str(obj[key])

# Converte o JSON para um DataFrame pandas
df_all_projects = pd.json_normalize(json_response)

# Printa o DataFrame parcialmente
df_all_projects.head()

,id,primary_site,dbgap_accession_number,project_id,disease_type,name,releasable,state,released,summary.file_count,summary.data_categories,summary.experimental_strategies,summary.case_count,summary.file_size,program.dbgap_accession_number,program.program_id,program.name
0,TARGET-AML,"[Unknown, Hematopoietic and reticuloendothelia...",phs000465,TARGET-AML,"[Not Applicable, Myeloid Leukemias]",Acute Myeloid Leukemia,True,open,True,51339,"[{'file_count': 12095, 'case_count': 806, 'dat...","[{'file_count': 28629, 'case_count': 2280, 'ex...",2492,104391798872383,phs000218,f1c391e9-8488-55a8-b777-302e786ea11d,TARGET
1,MATCH-Z1I,"[Bronchus and lung, Gallbladder, Pancreas, Unk...",phs002058,MATCH-Z1I,"[Squamous Cell Neoplasms, Epithelial Neoplasms...",Genomic Characterization CS-MATCH-0007 Arm Z1I,False,open,True,660,"[{'file_count': 327, 'case_count': 24, 'data_c...","[{'file_count': 374, 'case_count': 24, 'experi...",26,2396549656405,None,893b0820-b50b-5c75-85f5-3b028e251811,MATCH
2,HCMI-CMDC,"[Breast, Rectum, Nasal cavity and middle ear, ...",None,HCMI-CMDC,"[Cystic, Mucinous and Serous Neoplasms, Ductal...",NCI Cancer Model Development for the Human Can...,True,open,True,20761,"[{'file_count': 8450, 'case_count': 278, 'data...","[{'file_count': 8096, 'case_count': 276, 'expe...",278,130722797290059,phs001486,a5448c11-d46a-56aa-a5e1-5c1aa06404df,HCMI
3,MATCH-W,"[Breast, Renal pelvis, Corpus uteri, Bladder, ...",phs001948,MATCH-W,"[Cystic, Mucinous and Serous Neoplasms, Ductal...",Genomic Characterization CS-MATCH-0007 Arm W,False,open,True,1091,"[{'file_count': 535, 'case_count': 44, 'data_c...","[{'file_count': 614, 'case_count': 44, 'experi...",45,3929920960515,None,893b0820-b50b-5c75-85f5-3b028e251811,MATCH
4,MATCH-Z1D,"[Breast, Corpus uteri, Bones, joints and artic...",phs001859,MATCH-Z1D,"[Cystic, Mucinous and Serous Neoplasms, Ductal...",Genomic Characterization CS-MATCH-0007 Arm Z1D,False,open,True,891,"[{'file_count': 440, 'case_count': 34, 'data_c...","[{'file_count': 504, 'case_count': 34, 'experi...",36,3282605560716,None,893b0820-b50b-5c75-85f5-3b028e251811,MATCH


## Projetos Lançados com miRNA-Seq e RNA-Seq

In [7]:
# Filtra os projetos lançados pelo TCGA e remove as colunas desnecessárias
df_projects = df_all_projects \
    .query('released == "True"') \
    .drop(
        columns=[
            'id',
            'releasable',
            'released',
            'state',
            'summary.data_categories',
            'summary.file_count',
            'summary.file_size'
        ]
    )

# Inicializa listas para a contagem de casos com miRNA-Seq e RNA-Seq nos projetos
miRNA_Seq_case_count = [0] * df_projects.shape[0]
RNA_Seq_case_count = [0] * df_projects.shape[0]

# Preenche listas com a contagem de casos com miRNA-Seq e RNA-Seq em cada projeto
for index in range(df_projects.shape[0]):
    for info in df_projects['summary.experimental_strategies'][index]:
        if info['experimental_strategy'] in ['miRNA-Seq', 'RNA-Seq']:
            if info['experimental_strategy'] == 'miRNA-Seq':
                miRNA_Seq_case_count[index] = info['case_count']
            else:
                RNA_Seq_case_count[index] = info['case_count']

# Adiciona a contagem de casos de miRNA-Seq e RNA-Seq ao DataFrame
df_projects['miRNA-Seq_case_count'] = miRNA_Seq_case_count
df_projects['RNA-Seq_case_count'] = RNA_Seq_case_count

# Filtra os projetos com miRNA-Seq e RNA-Seq como estratégia experimental
df_projects = df_projects \
    .query('(`miRNA-Seq_case_count` > 0) & (`RNA-Seq_case_count` > 0)') \
    .drop(columns='summary.experimental_strategies') \
    .reset_index(drop=True)

# Reorganiza e renomeia as colunas do DataFrame
columns = [
    'project_id',
    'dbgap_accession_number',
    'name',
    'primary_site',
    'disease_type',
    'summary.case_count',
    'miRNA-Seq_case_count',
    'RNA-Seq_case_count',
    'program.program_id',
    'program.dbgap_accession_number',
    'program.name'
]
df_projects = df_projects[columns] \
    .rename(
        columns={
            'name': 'project_name',
            'summary.case_count': 'case_count',
            'program.program_id': 'program_id',
            'program.dbgap_accession_number': 'program_dbgap_accession_number',
            'program.name': 'program_name'
        }
    )

In [8]:
# Printa o DataFrame de projetos
pd.set_option('display.max_colwidth', 100)
df_projects

,project_id,dbgap_accession_number,project_name,primary_site,disease_type,case_count,miRNA-Seq_case_count,RNA-Seq_case_count,program_id,program_dbgap_accession_number,program_name
0,TARGET-AML,phs000465,Acute Myeloid Leukemia,"[Unknown, Hematopoietic and reticuloendothelial systems]","[Not Applicable, Myeloid Leukemias]",2492,1836,2280,f1c391e9-8488-55a8-b777-302e786ea11d,phs000218,TARGET
1,HCMI-CMDC,None,NCI Cancer Model Development for the Human Cancer Model Initiative,"[Breast, Rectum, Nasal cavity and middle ear, Bronchus and lung, Other and unspecified parts of ...","[Cystic, Mucinous and Serous Neoplasms, Ductal and Lobular Neoplasms, Adenomas and Adenocarcinom...",278,271,277,a5448c11-d46a-56aa-a5e1-5c1aa06404df,phs001486,HCMI
2,TCGA-PCPG,None,Pheochromocytoma and Paraganglioma,"[Other endocrine glands and related structures, Heart, mediastinum, and pleura, Connective, subc...",[Paragangliomas and Glomus Tumors],179,179,179,b80aa962-9650-5110-b3eb-bd087da808db,phs000178,TCGA
3,TCGA-THYM,None,Thymoma,"[Heart, mediastinum, and pleura, Thymus]",[Thymic Epithelial Neoplasms],124,124,120,b80aa962-9650-5110-b3eb-bd087da808db,phs000178,TCGA
4,TCGA-PAAD,None,Pancreatic Adenocarcinoma,[Pancreas],"[Cystic, Mucinous and Serous Neoplasms, Ductal and Lobular Neoplasms, Adenomas and Adenocarcinom...",185,178,178,b80aa962-9650-5110-b3eb-bd087da808db,phs000178,TCGA
5,TCGA-STAD,None,Stomach Adenocarcinoma,[Stomach],"[Cystic, Mucinous and Serous Neoplasms, Adenomas and Adenocarcinomas]",443,436,415,b80aa962-9650-5110-b3eb-bd087da808db,phs000178,TCGA
6,TCGA-TGCT,None,Testicular Germ Cell Tumors,[Testis],[Germ Cell Neoplasms],263,150,150,b80aa962-9650-5110-b3eb-bd087da808db,phs000178,TCGA
7,TCGA-SARC,None,Sarcoma,"[Bones, joints and articular cartilage of limbs, Uterus, NOS, Corpus uteri, Other and unspecifie...","[Lipomatous Neoplasms, Synovial-like Neoplasms, Fibromatous Neoplasms, Nerve Sheath Tumors, Myom...",261,259,259,b80aa962-9650-5110-b3eb-bd087da808db,phs000178,TCGA
8,TCGA-PRAD,None,Prostate Adenocarcinoma,[Prostate gland],"[Cystic, Mucinous and Serous Neoplasms, Adenomas and Adenocarcinomas, Ductal and Lobular Neoplasms]",500,494,497,b80aa962-9650-5110-b3eb-bd087da808db,phs000178,TCGA
9,TCGA-READ,None,Rectum Adenocarcinoma,"[Rectum, Rectosigmoid junction, Unknown, Colon, Connective, subcutaneous and other soft tissues]","[Cystic, Mucinous and Serous Neoplasms, Adenomas and Adenocarcinomas]",172,161,167,b80aa962-9650-5110-b3eb-bd087da808db,phs000178,TCGA


## Casos com miRNA-Seq e RNA-Seq

In [9]:
# Campos de interesse para a requisição ao endpoint
fields_cases = [
    'disease_type',
    'files.access',
    'files.data_category',
    'files.data_format',
    'files.data_type',
    'files.file_id',
    'files.experimental_strategy',
    'primary_site',
    'project.project_id',
]
fields_cases = ','.join(fields_cases)

# Inicializa o DataFrame de casos e arquivos
df_cases_and_files = pd.DataFrame()

# Busca pelos casos com miRNA-Seq e RNA-Seq de cada projeto de interesse
for index in range(df_projects.shape[0]):
    # Filtros empregados na requisição ao endpoint
    filters = {
        'op': 'and',
        'content': [
            {
                'op': '=',
                'content': {
                    'field': 'project.project_id',
                    'value': df_projects['project_id'][index]
                }
            },
            {
                'op': 'in',
                'content': {
                    'field': 'files.experimental_strategy',
                    'value': ['miRNA-Seq', 'RNA-Seq']
                }
            }
        ]
    }

    # Parâmetros para a requisição ao endpoint
    params = {
        'fields': fields_cases,
        'filters': filters,
        'size': str(df_projects['case_count'][index])
    }

    # Busca por todos os objetos do endpoint
    response = requests.post(
        url=CASES_ENDPOINT,
        headers={'Content-Type': 'application/json'},
        json=params
    )

    # Converte o conteúdo da resposta para JSON
    json_response = json.loads(response.content.decode('utf-8'))

    # Converte o JSON para um DataFrame pandas
    df_project_cases = pd.json_normalize(json_response['data']['hits'])

    # Concatena os casos deste projeto com os restantes
    if df_cases_and_files.empty == False:
        df_cases_and_files = pd.concat(
            [df_cases_and_files, df_project_cases], ignore_index=True
        )
    else:
        df_cases_and_files = df_project_cases.copy()

# Renomeia algumas colunas do DataFrame
df_cases_and_files = df_cases_and_files.rename(
    columns={'id': 'case_id', 'project.project_id': 'project_id'}
)

In [10]:
# Printa o DataFrame de casos e arquivos
df_cases_and_files

,case_id,primary_site,disease_type,files,project_id
0,58771370-5082-485e-ac68-13edfbd9ef0c,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,"[{'data_format': 'XLSX', 'access': 'open', 'file_id': '52fd584b-9ca3-4f7e-bdd5-fd9dce3d630b', 'd...",TARGET-AML
1,28da5b52-29ed-4817-a678-9206c5164da0,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,"[{'data_format': 'XLSX', 'access': 'open', 'file_id': '52fd584b-9ca3-4f7e-bdd5-fd9dce3d630b', 'd...",TARGET-AML
2,28dae019-4d93-42bf-9bb3-8e6e21bc1d29,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,"[{'data_format': 'TSV', 'access': 'controlled', 'file_id': '46894ff9-3571-4878-8134-d0ad10a3fd89...",TARGET-AML
3,28f7e68c-e0ab-4fb0-b835-2e506cb0012d,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,"[{'data_format': 'BAM', 'access': 'controlled', 'file_id': '87a249be-2899-4294-9fea-4a7578971f2d...",TARGET-AML
4,28ffdfb9-f051-4424-ad35-e6633e6bf42e,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,"[{'data_format': 'XLSX', 'access': 'open', 'file_id': '52fd584b-9ca3-4f7e-bdd5-fd9dce3d630b', 'd...",TARGET-AML
...,...,...,...,...,...
16606,a798a8cc-4e72-4a8c-9e20-74a14deafd12,Corpus uteri,Adenomas and Adenocarcinomas,"[{'data_format': 'BCR Biotab', 'access': 'open', 'file_id': '3eefa637-ec64-42fa-8d0d-c740de10666...",TCGA-UCEC
16607,fe2e89f7-8f4d-420a-a551-4877cf0fd1d3,Corpus uteri,Adenomas and Adenocarcinomas,"[{'data_format': 'SVS', 'access': 'open', 'file_id': '0d29b22e-3f4f-462a-bc73-c93237cb3867', 'da...",TCGA-UCEC
16608,db3a4986-55d5-4ecc-be73-59725dce3c33,Corpus uteri,Adenomas and Adenocarcinomas,"[{'data_format': 'MAF', 'access': 'controlled', 'file_id': 'eb6f7308-a02d-414c-b2c0-7c86a7651df5...",TCGA-UCEC
16609,ff6b5fc8-0572-4b58-b3a5-bcda41badbc8,Corpus uteri,"Cystic, Mucinous and Serous Neoplasms","[{'data_format': 'BAM', 'access': 'controlled', 'file_id': 'b0c765b4-4804-451c-812a-9cf2e86b570f...",TCGA-UCEC


## Arquivos com miRNA-Seq e RNA-Seq

In [11]:
# Explode as listas com dicionário sobre os arquivos dos casos
df_cases = df_cases_and_files.explode('files')

# Filtra os arquivos dos casos relacionados a miRNA-Seq e RNA-Seq
key = 'experimental_strategy'
df_cases = df_cases[
    df_cases['files'].apply(
        lambda x: (
            key in x and (x[key] == 'miRNA-Seq' or x[key] == 'RNA-Seq')
        )
    )
]

# Explode os dicionários com as informações sobre os arquivos de interesse
df_files = pd.json_normalize(df_cases['files'])

# Remove a coluna referente aos arquivos e redefine os índices do DataFrame
df_cases = df_cases.drop(columns='files').reset_index(drop=True)

# Concatena os DataFrames e define o DataFrame de arquivos
df_files = pd.concat([df_cases, df_files], axis='columns')
df_files = df_files.drop(columns=['primary_site', 'disease_type', 'project_id'])
df_files = df_files[[
    'file_id',
    'access',
    'data_format',
    'data_type',
    'data_category',
    'experimental_strategy',
    'case_id'
]]

# Define o DataFrame de casos
df_cases = df_cases \
    .drop_duplicates() \
    .reset_index(drop=True)

In [12]:
# Printa o DataFrame de casos
df_cases

,case_id,primary_site,disease_type,project_id
0,58771370-5082-485e-ac68-13edfbd9ef0c,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,TARGET-AML
1,28da5b52-29ed-4817-a678-9206c5164da0,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,TARGET-AML
2,28dae019-4d93-42bf-9bb3-8e6e21bc1d29,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,TARGET-AML
3,28f7e68c-e0ab-4fb0-b835-2e506cb0012d,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,TARGET-AML
4,28ffdfb9-f051-4424-ad35-e6633e6bf42e,Hematopoietic and reticuloendothelial systems,Myeloid Leukemias,TARGET-AML
...,...,...,...,...
16606,a798a8cc-4e72-4a8c-9e20-74a14deafd12,Corpus uteri,Adenomas and Adenocarcinomas,TCGA-UCEC
16607,fe2e89f7-8f4d-420a-a551-4877cf0fd1d3,Corpus uteri,Adenomas and Adenocarcinomas,TCGA-UCEC
16608,db3a4986-55d5-4ecc-be73-59725dce3c33,Corpus uteri,Adenomas and Adenocarcinomas,TCGA-UCEC
16609,ff6b5fc8-0572-4b58-b3a5-bcda41badbc8,Corpus uteri,"Cystic, Mucinous and Serous Neoplasms",TCGA-UCEC


In [13]:
# Printa o DataFrame de arquivos
df_files

,file_id,access,data_format,data_type,data_category,experimental_strategy,case_id
0,7e37b56e-4a47-4e14-aa0c-2317e31f3e53,controlled,TSV,Transcript Fusion,Structural Variation,RNA-Seq,58771370-5082-485e-ac68-13edfbd9ef0c
1,2b213562-4c34-4976-9563-ef2e581eb993,controlled,BAM,Aligned Reads,Sequencing Reads,RNA-Seq,58771370-5082-485e-ac68-13edfbd9ef0c
2,4188fa7a-7b11-47ae-ad63-701f5a2ffecb,controlled,BEDPE,Transcript Fusion,Structural Variation,RNA-Seq,58771370-5082-485e-ac68-13edfbd9ef0c
3,dd250c15-0be7-4243-b147-d4c872587844,controlled,BEDPE,Transcript Fusion,Structural Variation,RNA-Seq,58771370-5082-485e-ac68-13edfbd9ef0c
4,619cea2a-d035-4979-ad83-b7863eefc337,controlled,BAM,Aligned Reads,Sequencing Reads,RNA-Seq,58771370-5082-485e-ac68-13edfbd9ef0c
...,...,...,...,...,...,...,...
229791,d78ca382-fc44-48fb-b40e-3c56bab0ba99,open,TXT,miRNA Expression Quantification,Transcriptome Profiling,miRNA-Seq,43476c88-86bc-4a65-9327-f0d24a41c971
229792,af52e052-4b18-41f6-8647-8370405c4b00,controlled,TSV,Splice Junction Quantification,Transcriptome Profiling,RNA-Seq,43476c88-86bc-4a65-9327-f0d24a41c971
229793,f491ad0e-78db-4ed0-9d80-9d26bb0f736d,controlled,BAM,Aligned Reads,Sequencing Reads,miRNA-Seq,43476c88-86bc-4a65-9327-f0d24a41c971
229794,36cd1c9b-ab47-4a09-96d6-848cd59c08dd,open,TSV,Gene Expression Quantification,Transcriptome Profiling,RNA-Seq,43476c88-86bc-4a65-9327-f0d24a41c971
